In [6]:
import pandas as pd
import glob

all_files = glob.glob("out/bus/bus_*")
df_list = (pd.read_csv(f, header=None) for f in all_files)
df = pd.concat(df_list, ignore_index=True)

cols = df.items()
next(cols)

labeled_vehicle_positions = pd.DataFrame(columns=['timestamp', 'vehicle_id', 'route_id', 'coordinates'])

for col_name, col_data in cols:
    vehicle_data = col_data.str.split(pat=r"[\/\@;]", expand=True)
    vehicle_data.columns = ['vehicle_id', 'route_id', 'latitude', 'longitude']
    vehicle_data['coordinates'] = vehicle_data[['latitude', 'longitude']].apply(lambda a: tuple([float(a.iloc[1]), float(a.iloc[0])]), axis=1)
    vehicle_data.drop(['latitude', 'longitude'], axis=1, inplace=True)

    timestamps = df.iloc[:, 0]
    timestamps.name = 'timestamp'

    vehicle_positions = pd.concat([timestamps, vehicle_data], axis=1)
    labeled_vehicle_positions = pd.concat([labeled_vehicle_positions, vehicle_positions])

by_route = labeled_vehicle_positions.pivot(columns='route_id', index='timestamp')
south_campus = by_route.xs('38768', axis=1, level=1, drop_level=True)
green_housing = by_route.xs('38765', axis=1, level=1, drop_level=True)

In [8]:
from route import Route
from shapely import Point
import usu_routes

# GREEN_LINE_STOPS = {'Field House': (41.74445999, 111.8134483), 'Veterinary Science': (41.744432, 111.8110267), 'Industrial Science': (41.744343979, 111.808262326), 'Aggie Ice Cream': (41.745159696, 111.80430747), 'Lundstrum': (41.749617062, 111.802081881), 'Aggie Village 1': (41.750817743, 111.804389116), 'Aggie Village 2': (41.75307753, 111.806060476), 'Aggie Village 3': (41.751605001, 111.809274072), '800 E 900 N': (41.748601779, 111.814078833)}
# RED_LINE_STOPS = {'Taggart Student Center': (41.743714774, 111.813960182), 'Lundstrum': (41.749617062, 111.802081881), 'The Bull': (41.750053756, 111.813282805), 'Logan Cemetery': (41.749962755, 111.805868979), 'Fine Arts Bldg': (41.74344203, 111.8043394), 'Family Life Bldg': (41.74007151, 111.811128394), '670 E 500 N': (41.74060939, 111.8176719)}

AT_STOP_THRESHOLD_M = 16

green_line_segments = usu_routes.import_route(usu_routes.GREEN_ROUTE)
green_line = Route(green_line_segments)

def dist_marker(coords, route: Route):
    point = Point(coords)
    return tuple(route.distance_marker_for(point))[0]

green_housing["stop"] = green_housing['coordinates'].apply(dist_marker, args=(green_line,))
green_housing.to_csv("out/bus/green_housing.csv")

# south_campus["stop"] = south_campus['coordinates'].apply(stop_name, args=(RED_LINE_STOPS,))
# south_campus.to_csv("out/bus/south_campus.csv")